In [11]:
import json
import time
import hashlib
import ipaddress
import requests
from kafka import KafkaProducer

KAFKA_BROKER = "192.168.137.119"  # Ganti dengan IP PC Master
TOPIC_NAME = "wikimedia-recentchange"
STREAM_URL = "https://stream.wikimedia.org/v2/stream/recentchange"

HEADERS = {
    "User-Agent": "AmikomBigDataUTS-ST153/1.0 (kelompok-bigdata@students.amikom.ac.id)"
}

def samarkan_ip(user_str):
    if not user_str:
        return user_str
    try:
        ipaddress.ip_address(user_str)
        return hashlib.sha256(user_str.encode("utf-8")).hexdigest()[:16]
    except ValueError:
        return user_str

producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BROKER],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
    acks="all",
    retries=5
)

def jalankan_producer():
    total_pesan = 0
    waktu_mulai = time.time()

    while True:
        try:
            print("Menyambungkan ke Wikimedia EventStreams...")
            with requests.get(STREAM_URL, headers=HEADERS, stream=True, timeout=30) as resp:
                resp.raise_for_status()
                for line in resp.iter_lines(decode_unicode=True):
                    if line and line.startswith("data: "):
                        try:
                            raw = json.loads(line[6:])
                            length_data = raw.get("length") or {}

                            data_bersih = {
                                "id": raw.get("id"),
                                "type": raw.get("type"),
                                "wiki": raw.get("wiki"),
                                "server_name": raw.get("server_name"),
                                "title": raw.get("title"),
                                "namespace": raw.get("namespace"),
                                "user": samarkan_ip(raw.get("user")),
                                "bot": raw.get("bot"),
                                "minor": raw.get("minor"),
                                "timestamp": raw.get("timestamp"),
                                "length": {
                                    "old": length_data.get("old"),
                                    "new": length_data.get("new")
                                },
                                "comment": raw.get("comment")
                            }

                            msg_key = f"{data_bersih['wiki']}_{data_bersih['id']}"
                            producer.send(TOPIC_NAME, key=msg_key, value=data_bersih)
                            total_pesan += 1

                            if total_pesan % 500 == 0:
                                durasi = time.time() - waktu_mulai
                                laju = total_pesan / durasi if durasi > 0 else 0
                                print(f"Terkirim: {total_pesan} pesan | Durasi: {durasi:.1f}s | Laju: {laju:.2f} pesan/detik")

                        except json.JSONDecodeError:
                            continue
        except Exception as e:
            print(f"Koneksi terputus ({e}), menyambung ulang dalam 5 detik...")
            time.sleep(5)

if __name__ == "__main__":
    try:
        jalankan_producer()
    except KeyboardInterrupt:
        print("\nProducer dihentikan.")
    finally:
        producer.flush()
        producer.close()

Menyambungkan ke Wikimedia EventStreams...
Terkirim: 500 pesan | Durasi: 11.4s | Laju: 43.84 pesan/detik
Terkirim: 1000 pesan | Durasi: 21.4s | Laju: 46.66 pesan/detik
Terkirim: 1500 pesan | Durasi: 31.4s | Laju: 47.70 pesan/detik
Terkirim: 2000 pesan | Durasi: 40.8s | Laju: 49.03 pesan/detik
Terkirim: 2500 pesan | Durasi: 51.3s | Laju: 48.71 pesan/detik
Terkirim: 3000 pesan | Durasi: 61.8s | Laju: 48.54 pesan/detik
Terkirim: 3500 pesan | Durasi: 69.5s | Laju: 50.36 pesan/detik
Terkirim: 4000 pesan | Durasi: 80.2s | Laju: 49.88 pesan/detik
Terkirim: 4500 pesan | Durasi: 88.6s | Laju: 50.81 pesan/detik
Terkirim: 5000 pesan | Durasi: 99.2s | Laju: 50.40 pesan/detik
Terkirim: 5500 pesan | Durasi: 108.9s | Laju: 50.51 pesan/detik
Terkirim: 6000 pesan | Durasi: 118.2s | Laju: 50.77 pesan/detik
Terkirim: 6500 pesan | Durasi: 128.6s | Laju: 50.56 pesan/detik
Terkirim: 7000 pesan | Durasi: 138.4s | Laju: 50.56 pesan/detik
Terkirim: 7500 pesan | Durasi: 149.6s | Laju: 50.12 pesan/detik
Terkirim

In [14]:
import os
import json
from kafka import KafkaConsumer

KAFKA_BROKER = "192.168.137.119:9092"
TOPIC_NAME = "wikimedia-recentchange"
OUTPUT_FILE = r"C:\shared\wikimedia_snapshot.jsonl"
MAX_PESAN = 200000  # Batas maksimal pesan yang ditarik

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=[KAFKA_BROKER],
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    consumer_timeout_ms=5000
)

count = 0
print("Mulai menarik snapshot dari Kafka...")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for msg in consumer:
        f.write(msg.value.decode("utf-8") + "\n")
        count += 1
        
        if count % 5000 == 0:
            print(f"Tersimpan: {count} pesan...")
            
        if count >= MAX_PESAN:
            print("Target pesan tercapai, penarikan dihentikan.")
            break

consumer.close()
print(f"Snapshot selesai disimpan: {count} pesan di {OUTPUT_FILE}")

Mulai menarik snapshot dari Kafka...
Tersimpan: 5000 pesan...
Tersimpan: 10000 pesan...
Tersimpan: 15000 pesan...
Tersimpan: 20000 pesan...
Tersimpan: 25000 pesan...
Tersimpan: 30000 pesan...
Tersimpan: 35000 pesan...
Tersimpan: 40000 pesan...
Tersimpan: 45000 pesan...
Tersimpan: 50000 pesan...
Tersimpan: 55000 pesan...
Tersimpan: 60000 pesan...
Tersimpan: 65000 pesan...
Tersimpan: 70000 pesan...
Tersimpan: 75000 pesan...
Tersimpan: 80000 pesan...
Tersimpan: 85000 pesan...
Tersimpan: 90000 pesan...
Tersimpan: 95000 pesan...
Tersimpan: 100000 pesan...
Tersimpan: 105000 pesan...
Tersimpan: 110000 pesan...
Tersimpan: 115000 pesan...
Tersimpan: 120000 pesan...
Tersimpan: 125000 pesan...
Tersimpan: 130000 pesan...
Tersimpan: 135000 pesan...
Tersimpan: 140000 pesan...
Tersimpan: 145000 pesan...
Tersimpan: 150000 pesan...
Tersimpan: 155000 pesan...
Tersimpan: 160000 pesan...
Tersimpan: 165000 pesan...
Tersimpan: 170000 pesan...
Tersimpan: 175000 pesan...
Tersimpan: 180000 pesan...
Tersimpan: 

In [15]:
import json
import re
import time
from pyspark.sql import SparkSession

MASTER_URL = "spark://192.168.137.119:7077"
SNAPSHOT_PATH = r"C:\shared\wikimedia_snapshot.jsonl"

spark = (
    SparkSession.builder
    .appName("UTS_Soal2_Spark_RDD_Cluster")
    .master(MASTER_URL)
    .config("spark.driver.host", "192.168.137.119")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
sc = spark.sparkContext
print("Terhubung ke Spark Master:", sc.master)

# Baru lanjut ke kode Soal 2a (raw_rdd = sc.textFile(...)) di bawahnya

# ==========================================
# SOAL 2a: Parsing JSON, Accumulator & glom()
# ==========================================
raw_rdd = sc.textFile(SNAPSHOT_PATH, minPartitions=4)
bad_records_acc = sc.accumulator(0)

def parse_json_safe(line):
    try:
        data = json.loads(line)
        if isinstance(data, dict) and "wiki" in data:
            return data
        bad_records_acc.add(1)
        return None
    except Exception:
        bad_records_acc.add(1)
        return None

parsed_rdd = raw_rdd.map(parse_json_safe).filter(lambda x: x is not None)

total_valid = parsed_rdd.count()
num_partitions = parsed_rdd.getNumPartitions()
elemen_per_partisi = parsed_rdd.glom().map(len).collect()

print("=== HASIL SOAL 2a ===")
print(f"Total data valid       : {total_valid}")
print(f"Total data rusak (acc) : {bad_records_acc.value}")
print(f"Jumlah partisi         : {num_partitions}")
print(f"Sebaran per partisi    : {elemen_per_partisi}\n")

# ==========================================
# SOAL 2b: Operasi RDD (Narrow vs Wide)
# ==========================================
# (i) 10 wiki dengan edit manusia terbanyak
human_edit_rdd = (
    parsed_rdd
    .filter(lambda x: x.get("type") == "edit" and x.get("bot") is False)  # NARROW
    .map(lambda x: (x.get("wiki"), 1))                                    # NARROW
    .reduceByKey(lambda a, b: a + b)                                      # WIDE
)
top_10_wiki = human_edit_rdd.takeOrdered(10, key=lambda x: -x[1])         # ACTION

print("=== SOAL 2b (i): Top 10 Wiki Edit Manusia ===")
for wiki, jml in top_10_wiki:
    print(f"{wiki:<20} : {jml}")

# (ii) 10 halaman paling banyak disunting pada idwiki
idwiki_pages_rdd = (
    parsed_rdd
    .filter(lambda x: x.get("wiki") == "idwiki" and x.get("type") == "edit")  # NARROW
    .map(lambda x: (x.get("title"), 1))                                       # NARROW
    .reduceByKey(lambda a, b: a + b)                                          # WIDE
)
top_10_idwiki_pages = idwiki_pages_rdd.takeOrdered(10, key=lambda x: -x[1])   # ACTION

print("\n=== SOAL 2b (ii): Top 10 Halaman idwiki ===")
for title, jml in top_10_idwiki_pages:
    print(f"{title:<35} : {jml}")

# (iii) 15 kata terbanyak pada comment penyuntingan idwiki
stopwords_id = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "pada", "dengan",
    "atau", "sebagai", "dalam", "oleh", "adalah", "telah", "sudah", "akan",
    "tidak", "enggak", "bisa", "ada", "juga", "lebih", "saat", "bagian",
    "suntingan", "halaman", "kategori", "pranala", "rujukan", "lihat", "wp",
    "ref", "http", "https", "www", "com", "org", "id", "wikipedia"
}

def tokenize_clean(comment_text):
    if not comment_text:
        return []
    words = re.findall(r"[a-zA-Z]{3,}", comment_text.lower())
    return [w for w in words if w not in stopwords_id]

idwiki_words_rdd = (
    parsed_rdd
    .filter(lambda x: x.get("wiki") == "idwiki" and x.get("type") == "edit")  # NARROW
    .flatMap(lambda x: tokenize_clean(x.get("comment")))                      # NARROW
    .map(lambda w: (w, 1))                                                    # NARROW
    .reduceByKey(lambda a, b: a + b)                                          # WIDE
)
top_15_words = idwiki_words_rdd.takeOrdered(15, key=lambda x: -x[1])          # ACTION

print("\n=== SOAL 2b (iii): Top 15 Kata Comment idwiki ===")
for word, jml in top_15_words:
    print(f"{word:<20} : {jml}")

# ==========================================
# SOAL 2c: Benchmark reduceByKey vs groupByKey
# ==========================================
t0 = time.time()
_ = (
    parsed_rdd
    .filter(lambda x: x.get("type") == "edit" and x.get("bot") is False)
    .map(lambda x: (x.get("wiki"), 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)
waktu_reduce = time.time() - t0

t1 = time.time()
_ = (
    parsed_rdd
    .filter(lambda x: x.get("type") == "edit" and x.get("bot") is False)
    .map(lambda x: (x.get("wiki"), 1))
    .groupByKey()
    .mapValues(sum)
    .collect()
)
waktu_group = time.time() - t1

print("\n=== HASIL SOAL 2c: Benchmark Shuffle ===")
print(f"Waktu reduceByKey : {waktu_reduce:.4f} detik")
print(f"Waktu groupByKey  : {waktu_group:.4f} detik")

# ==========================================
# SOAL 2d: Lineage & Pengaruh cache()
# ==========================================
print("\n=== HASIL SOAL 2d: Lineage human_edit_rdd ===")
print(human_edit_rdd.toDebugString().decode("utf-8"))

parsed_rdd.unpersist()
t_no_cache = time.time()
_ = parsed_rdd.filter(lambda x: x.get("type") == "edit").count()
_ = parsed_rdd.filter(lambda x: x.get("bot") is True).count()
_ = parsed_rdd.filter(lambda x: x.get("wiki") == "idwiki").count()
durasi_no_cache = time.time() - t_no_cache

parsed_rdd.cache()
_ = parsed_rdd.count()  # Materialisasi ke memori Worker

t_with_cache = time.time()
_ = parsed_rdd.filter(lambda x: x.get("type") == "edit").count()
_ = parsed_rdd.filter(lambda x: x.get("bot") is True).count()
_ = parsed_rdd.filter(lambda x: x.get("wiki") == "idwiki").count()
durasi_with_cache = time.time() - t_with_cache

print(f"\nWaktu 3 action TANPA cache() : {durasi_no_cache:.4f} detik")
print(f"Waktu 3 action DENGAN cache(): {durasi_with_cache:.4f} detik")

Terhubung ke Spark Master: spark://192.168.137.119:7077
=== HASIL SOAL 2a ===
Total data valid       : 199997
Total data rusak (acc) : 0
Jumlah partisi         : 4
Sebaran per partisi    : [50323, 50521, 49436, 49717]

=== SOAL 2b (i): Top 10 Wiki Edit Manusia ===
wikidatawiki         : 42106
commonswiki          : 6565
enwiki               : 6516
dewiki               : 1867
frwiki               : 1510
trwiki               : 1441
itwiki               : 1258
ruwiki               : 948
jawiki               : 809
zhwiki               : 612

=== SOAL 2b (ii): Top 10 Halaman idwiki ===
Pengguna:YTTA12/bak pasir           : 7
Zara Zettira                        : 7
Pengguna:RamaBeni2011/Bak pasir     : 7
Yuke Yurike                         : 5
Unjuk rasa November 2026 di Indonesia : 4
Orang Hindu Sindhi                  : 4
Ramodatodes                         : 4
Pengguna:LiškovBot/Switch           : 3
Universitas Islam Riau              : 3
Hendro Siswanto                     : 3

=== SOAL 

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, BooleanType
)

MASTER_URL = "spark://192.168.137.119:7077"
SNAPSHOT_PATH = r"C:\shared\wikimedia_snapshot.jsonl"

spark = (
    SparkSession.builder
    .appName("UTS_Soal3_Spark_SQL_Cluster")
    .master(MASTER_URL)
    .config("spark.driver.host", "192.168.137.119")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)
sc = spark.sparkContext

# 1. Definisi skema eksplisit sesuai struktur JSON Wikipedia[cite: 2, 3]
schema_wikimedia = StructType([
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("title", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("timestamp", LongType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True),
    StructField("comment", StringType(), True)
])

# 2. Baca file di Master lalu distribusikan langsung ke memori Worker
with open(SNAPSHOT_PATH, "r", encoding="utf-8") as f:
    raw_lines = [line.strip() for line in f if line.strip()]

rdd_lines = sc.parallelize(raw_lines, numSlices=16)

df_raw = (
    spark.createDataFrame(rdd_lines, StringType())
    .select(F.from_json(F.col("value"), schema_wikimedia).alias("data"))
    .select("data.*")
)

# 3. Ubah timestamp ke tipe timestamp & tambah kolom selisih_byte[cite: 3]
df_wikimedia = (
    df_raw
    .withColumn("event_time", F.timestamp_seconds(F.col("timestamp")))
    .withColumn("selisih_byte", F.col("length.new") - F.coalesce(F.col("length.old"), F.lit(0)))
    .cache()
)

# 4. Daftarkan sebagai temporary view[cite: 3]
df_wikimedia.createOrReplaceTempView("wikimedia_events")

print("=== Skema DataFrame Eksplisit ===")
df_wikimedia.printSchema()
df_wikimedia.select("wiki", "type", "user", "bot", "event_time", "selisih_byte").show(5, truncate=False)

=== Skema DataFrame Eksplisit ===
root
 |-- id: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- user: string (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: long (nullable = true)
 |    |-- new: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- selisih_byte: long (nullable = true)

+------------+----------+--------------+-----+-------------------+------------+
|wiki        |type      |user          |bot  |event_time         |selisih_byte|
+------------+----------+--------------+-----+-------------------+------------+
|wikidatawiki|edit      |Mahlathamnyama|false|2026-11-20 17:02:05|114         |
|ruwiktionary|categorize|

In [17]:
# (i) Jumlah edit per menit pada 5 wiki teratas
print("=== 3b (i): Jumlah Edit per Menit pada 5 Wiki Teratas ===")
q_3b_1 = spark.sql("""
    WITH top5_wiki AS (
        SELECT wiki
        FROM wikimedia_events
        WHERE type = 'edit' AND wiki IS NOT NULL
        GROUP BY wiki
        ORDER BY COUNT(*) DESC
        LIMIT 5
    )
    SELECT
        date_trunc('minute', e.event_time) AS menit,
        e.wiki,
        COUNT(*) AS jumlah_edit
    FROM wikimedia_events e
    JOIN top5_wiki t ON e.wiki = t.wiki
    WHERE e.type = 'edit' AND e.event_time IS NOT NULL
    GROUP BY date_trunc('minute', e.event_time), e.wiki
    ORDER BY menit ASC, jumlah_edit DESC
""")
q_3b_1.show(15, truncate=False)

# (ii) Persentase bot dan manusia per wiki
print("=== 3b (ii): Persentase Bot dan Manusia per Wiki ===")
q_3b_2 = spark.sql("""
    SELECT
        wiki,
        COUNT(*) AS total_aktivitas,
        SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) AS jml_bot,
        SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) AS jml_manusia,
        ROUND(SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS persen_bot,
        ROUND(SUM(CASE WHEN bot = false THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS persen_manusia
    FROM wikimedia_events
    WHERE wiki IS NOT NULL
    GROUP BY wiki
    ORDER BY total_aktivitas DESC
""")
q_3b_2.show(10, truncate=False)

# Cek konsistensi dengan Soal 2b (i) (khusus type='edit' dan bot=false)[cite: 2, 3]
print("=== Verifikasi Konsistensi dengan Hasil RDD Soal 2b (i) ===")
spark.sql("""
    SELECT wiki, COUNT(*) AS edit_manusia_sql
    FROM wikimedia_events
    WHERE type = 'edit' AND bot = false
    GROUP BY wiki
    ORDER BY edit_manusia_sql DESC
    LIMIT 10
""").show(10, truncate=False)

# (iii) Rata-rata dan median (percentile_approx) abs(selisih_byte) per type[cite: 3]
print("=== 3b (iii): Rata-rata dan Median abs(selisih_byte) per Type ===")
q_3b_3 = spark.sql("""
    SELECT
        type,
        COUNT(*) AS jumlah_data,
        ROUND(AVG(ABS(selisih_byte)), 2) AS rata_rata_abs_byte,
        percentile_approx(ABS(selisih_byte), 0.5) AS median_abs_byte
    FROM wikimedia_events
    WHERE type IS NOT NULL AND selisih_byte IS NOT NULL
    GROUP BY type
    ORDER BY rata_rata_abs_byte DESC
""")
q_3b_3.show(truncate=False)

# (iv) 10 halaman idwiki dengan jumlah editor unik terbanyak[cite: 3]
print("=== 3b (iv): Top 10 Halaman idwiki dengan Editor Unik Terbanyak ===")
q_3b_4 = spark.sql("""
    SELECT
        title,
        COUNT(DISTINCT user) AS jumlah_editor_unik,
        COUNT(*) AS total_edit
    FROM wikimedia_events
    WHERE wiki = 'idwiki' AND type = 'edit' AND title IS NOT NULL
    GROUP BY title
    ORDER BY jumlah_editor_unik DESC, total_edit DESC
    LIMIT 10
""")
q_3b_4.show(10, truncate=False)

=== 3b (i): Jumlah Edit per Menit pada 5 Wiki Teratas ===
+-------------------+------------+-----------+
|menit              |wiki        |jumlah_edit|
+-------------------+------------+-----------+
|2026-11-20 17:01:00|enwiki      |2          |
|2026-11-20 17:02:00|wikidatawiki|363        |
|2026-11-20 17:02:00|commonswiki |289        |
|2026-11-20 17:02:00|enwiki      |119        |
|2026-11-20 17:02:00|frwiki      |20         |
|2026-11-20 17:02:00|itwiki      |7          |
|2026-11-20 17:03:00|commonswiki |442        |
|2026-11-20 17:03:00|wikidatawiki|416        |
|2026-11-20 17:03:00|enwiki      |52         |
|2026-11-20 17:03:00|itwiki      |27         |
|2026-11-20 17:03:00|frwiki      |24         |
|2026-11-20 17:04:00|wikidatawiki|386        |
|2026-11-20 17:04:00|commonswiki |250        |
|2026-11-20 17:04:00|enwiki      |60         |
|2026-11-20 17:04:00|itwiki      |25         |
+-------------------+------------+-----------+
only showing top 15 rows
=== 3b (ii): Persentase 

In [18]:
# Hitung jumlah edit per halaman dalam jendela 5 menit[cite: 3]
window_edit_df = (
    df_wikimedia
    .filter((F.col("type") == "edit") & F.col("event_time").isNotNull() & F.col("title").isNotNull())
    .groupBy("wiki", "title", F.window("event_time", "5 minutes").alias("jendela_5_menit"))
    .agg(F.count("*").alias("jumlah_edit"))
)

# Hitung distribusi persentil untuk menentukan k[cite: 3]
kuantil = window_edit_df.approxQuantile("jumlah_edit", [0.50, 0.90, 0.95, 0.99], 0.01)
p50, p90, p95, p99 = kuantil
k_threshold = int(round(p99))

print("=== Distribusi Edit per Halaman dalam Jendela 5 Menit ===")
print(f"Persentil ke-50 (Median) : {p50}")
print(f"Persentil ke-90          : {p90}")
print(f"Persentil ke-95          : {p95}")
print(f"Persentil ke-99 (nilai k): {k_threshold}\n")

# Filter halaman yang menerima >= k edit dalam satu jendela 5 menit[cite: 3]
lonjakan_df = (
    window_edit_df
    .filter(F.col("jumlah_edit") >= k_threshold)
    .select(
        "wiki",
        "title",
        F.col("jendela_5_menit.start").alias("waktu_mulai"),
        F.col("jendela_5_menit.end").alias("waktu_selesai"),
        "jumlah_edit"
    )
    .orderBy(F.col("jumlah_edit").desc())
)

print(f"=== Halaman Terdeteksi Lonjakan (>= {k_threshold} edit / 5 menit) ===")
lonjakan_df.show(15, truncate=False)

=== Distribusi Edit per Halaman dalam Jendela 5 Menit ===
Persentil ke-50 (Median) : 1.0
Persentil ke-90          : 1.0
Persentil ke-95          : 2.0
Persentil ke-99 (nilai k): 68

=== Halaman Terdeteksi Lonjakan (>= 68 edit / 5 menit) ===
+------------+----------+-------------------+-------------------+-----------+
|wiki        |title     |waktu_mulai        |waktu_selesai      |jumlah_edit|
+------------+----------+-------------------+-------------------+-----------+
|wikidatawiki|Q141047692|2026-11-20 17:40:00|2026-11-20 17:45:00|68         |
+------------+----------+-------------------+-------------------+-----------+



In [19]:
import time

query_uji = """
    SELECT
        wiki,
        COUNT(*) AS total_event,
        SUM(CASE WHEN bot = true THEN 1 ELSE 0 END) AS jml_bot,
        ROUND(AVG(ABS(selisih_byte)), 2) AS avg_byte
    FROM wikimedia_events
    WHERE wiki IS NOT NULL
    GROUP BY wiki
    ORDER BY total_event DESC
"""

# 1. Tampilkan Physical Plan menggunakan explain()[cite: 3]
print("=== Physical Plan (explain) ===")
spark.sql(query_uji).explain()

# 2. Ukur waktu SEBELUM optimasi (default 200 shuffle partitions)[cite: 3]
spark.conf.set("spark.sql.shuffle.partitions", "200")
t0 = time.time()
_ = spark.sql(query_uji).collect()
waktu_sebelum = time.time() - t0

# 3. Ukur waktu SESUDAH optimasi (16 shuffle partitions sesuai jumlah core cluster)[cite: 3, 11]
spark.conf.set("spark.sql.shuffle.partitions", "16")
t1 = time.time()
_ = spark.sql(query_uji).collect()
waktu_sesudah = time.time() - t1

print("\n=== Hasil Perbandingan Optimasi Shuffle Partitions ===")
print(f"Sebelum optimasi (200 partisi) : {waktu_sebelum:.4f} detik")
print(f"Sesudah optimasi (16 partisi)  : {waktu_sesudah:.4f} detik")

=== Physical Plan (explain) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [total_event#9650L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(total_event#9650L DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=2979]
      +- HashAggregate(keys=[wiki#6258], functions=[count(1), sum(CASE WHEN bot#6263 THEN 1 ELSE 0 END), avg(abs(selisih_byte#6269L))])
         +- Exchange hashpartitioning(wiki#6258, 8), ENSURE_REQUIREMENTS, [plan_id=2976]
            +- HashAggregate(keys=[wiki#6258], functions=[partial_count(1), partial_sum(CASE WHEN bot#6263 THEN 1 ELSE 0 END), partial_avg(abs(selisih_byte#6269L))])
               +- Filter isnotnull(wiki#6258)
                  +- InMemoryTableScan [wiki#6258, bot#6263, selisih_byte#6269L], [isnotnull(wiki#6258)]
                        +- InMemoryRelation [id#6256L, type#6257, wiki#6258, server_name#6259, title#6260, namespace#6261, user#6262, bot#6263, minor#6264, timestamp#6265L, length#6266, comment#6267, event_tim

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, BooleanType
)

MASTER_URL = "spark://192.168.137.119:7077"
KAFKA_BROKER = "192.168.137.119:9092"
TOPIC_NAME = "wikimedia-recentchange"
CHECKPOINT_10S = r"C:\shared\checkpoint_soal4_10s"
DASHBOARD_CSV = r"C:\shared\dashboard_streaming.csv"

os.makedirs(r"C:\shared", exist_ok=True)
if os.path.exists(DASHBOARD_CSV):
    os.remove(DASHBOARD_CSV)

spark = (
    SparkSession.builder
    .appName("UTS_Soal4_Structured_Streaming")
    .master(MASTER_URL)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.3")
    .config("spark.driver.host", "192.168.137.119")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# 1. Skema JSON untuk from_json
schema_stream = StructType([
    StructField("id", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("timestamp", LongType(), True)
])

# 2. Baca topic Kafka sebagai stream
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "latest")
    .load()
)

# 3. Urai JSON dengan from_json dan jadikan timestamp sebagai event_time
parsed_stream = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), schema_stream).alias("data"))
    .select("data.*")
    .filter(F.col("type") == "edit")
    .withColumn("event_time", F.timestamp_seconds(F.col("timestamp")))
)

# 4. Hitung jumlah edit & rasio bot per jendela 1 menit dengan watermark 2 menit
agg_stream = (
    parsed_stream
    .withWatermark("event_time", "2 minutes")
    .groupBy(
        F.window(F.col("event_time"), "1 minute").alias("jendela_waktu"),
        F.col("wiki")
    )
    .agg(
        F.count("*").alias("jumlah_edit"),
        F.sum(F.when(F.col("bot") == True, 1).otherwise(0)).alias("jumlah_bot"),
        F.round(F.sum(F.when(F.col("bot") == True, 1).otherwise(0)) / F.count("*"), 4).alias("rasio_bot")
    )
    .select(
        F.col("jendela_waktu.start").alias("window_start"),
        F.col("jendela_waktu.end").alias("window_end"),
        "wiki",
        "jumlah_edit",
        "jumlah_bot",
        "rasio_bot"
    )
)

# 5. Fungsi foreachBatch untuk output console dan penyimpanan file dashboard
def proses_tiap_batch(batch_df, batch_id):
    jumlah_baris = batch_df.count()
    print(f"\n=== Batch ID: {batch_id} | Jumlah Grup Wiki: {jumlah_baris} ===")
    if jumlah_baris > 0:
        hasil_urut = batch_df.orderBy(F.col("jumlah_edit").desc())
        hasil_urut.show(10, truncate=False)
        
        pdf = hasil_urut.toPandas()
        pdf["batch_id"] = batch_id
        tulis_header = not os.path.exists(DASHBOARD_CSV)
        pdf.to_csv(DASHBOARD_CSV, mode="a", header=tulis_header, index=False)

# 6. Jalankan query dengan checkpointLocation dan trigger 10 detik
query_10s = (
    agg_stream.writeStream
    .outputMode("update")
    .foreachBatch(proses_tiap_batch)
    .option("checkpointLocation", CHECKPOINT_10S)
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming berjalan dengan trigger 10 detik... Biarkan berjalan selama 1-2 menit.")

Streaming berjalan dengan trigger 10 detik... Biarkan berjalan selama 1-2 menit.


In [2]:
import time

def cetak_metrik_progress(query_obj, label_trigger):
    prog = query_obj.lastProgress
    if prog:
        print(f"=== Metrik Kinerja ({label_trigger}) ===")
        print(f"Batch ID                : {prog.get('batchId')}")
        print(f"Num Input Rows          : {prog.get('numInputRows')} baris")
        print(f"inputRowsPerSecond      : {prog.get('inputRowsPerSecond'):.2f} baris/detik")
        print(f"processedRowsPerSecond  : {prog.get('processedRowsPerSecond'):.2f} baris/detik")
        print(f"Durasi Batch Total      : {prog.get('batchDuration')} ms")
        print(f"Durasi Eksekusi Trigger : {prog.get('durationMs', {}).get('triggerExecution')} ms\n")
    else:
        print(f"Belum ada progress tercatat untuk {label_trigger}.\n")

# 1. Ambil metrik dari query trigger 10 detik
cetak_metrik_progress(query_10s, "Trigger 10 Detik")
query_10s.stop()

# 2. Jalankan pengujian kedua dengan trigger 30 detik
CHECKPOINT_30S = r"C:\shared\checkpoint_soal4_30s"

query_30s = (
    agg_stream.writeStream
    .outputMode("update")
    .foreachBatch(proses_tiap_batch)
    .option("checkpointLocation", CHECKPOINT_30S)
    .trigger(processingTime="30 seconds")
    .start()
)

print("Menjalankan trigger 30 detik selama 70 detik (sekitar 2 batch)...")
time.sleep(70)

cetak_metrik_progress(query_30s, "Trigger 30 Detik")
query_30s.stop()

Belum ada progress tercatat untuk Trigger 10 Detik.

Menjalankan trigger 30 detik selama 70 detik (sekitar 2 batch)...

=== Batch ID: 0 | Jumlah Grup Wiki: 1 ===
+-------------------+-------------------+-----------+-----------+----------+---------+
|window_start       |window_end         |wiki       |jumlah_edit|jumlah_bot|rasio_bot|
+-------------------+-------------------+-----------+-----------+----------+---------+
|2026-11-21 04:56:00|2026-11-21 04:57:00|commonswiki|1          |1         |1.0      |
+-------------------+-------------------+-----------+-----------+----------+---------+

=== Metrik Kinerja (Trigger 30 Detik) ===
Batch ID                : 0
Num Input Rows          : 2 baris
inputRowsPerSecond      : 0.00 baris/detik
processedRowsPerSecond  : 0.00 baris/detik
Durasi Batch Total      : 48332 ms
Durasi Eksekusi Trigger : 48321 ms



In [3]:
print("=== Memulai Ulang Stream dengan Checkpoint yang Sama ===")
print(f"Lokasi Checkpoint: {CHECKPOINT_10S}")

query_restart = (
    agg_stream.writeStream
    .outputMode("update")
    .foreachBatch(proses_tiap_batch)
    .option("checkpointLocation", CHECKPOINT_10S)
    .trigger(processingTime="10 seconds")
    .start()
)

# Biarkan memproses antrean yang tertinggal selama 35 detik
time.sleep(35)
cetak_metrik_progress(query_restart, "Setelah Restart Checkpoint")
query_restart.stop()
print("Pengujian Soal 4 selesai.")

=== Memulai Ulang Stream dengan Checkpoint yang Sama ===
Lokasi Checkpoint: C:\shared\checkpoint_soal4_10s

=== Batch ID: 0 | Jumlah Grup Wiki: 7 ===
+-------------------+-------------------+------------+-----------+----------+---------+
|window_start       |window_end         |wiki        |jumlah_edit|jumlah_bot|rasio_bot|
+-------------------+-------------------+------------+-----------+----------+---------+
|2026-11-21 04:52:00|2026-11-21 04:53:00|wikidatawiki|3          |0         |0.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|zhwiki      |2          |2         |1.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|commonswiki |2          |2         |1.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|arwiki      |1          |1         |1.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|ttwiki      |1          |1         |1.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|bnwiki      |1          |1         |1.0      |
|2026-11-21 04:52:00|2026-11-21 04:53:00|plwiki      |1   